In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# Load data
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

# Preprocessing
train_df["Year_Num"] = train_df["Year"].str.replace("x", "0").astype(int)
train_df_model_next = train_df.dropna(subset=["Ladder_Position_Next"])

# Feature selection
features = [
    "Games_Won", "Games_Lost", "Games_Drawn", "Points_For", "Points_Against",
    "Percentage", "Ladder_Position", "Average_Age", "Ladder_Position_Prev",
    "Close_Games_Won", "Close_Games_Lost", "Away_Games_Won", "Last_Five",
    "Average_Attendance", "Players_Used", "Victorian"
]

# Prepare training data
X_train = train_df_model_next[features].copy()
X_train["Victorian"] = X_train["Victorian"].astype(int)
y_train = train_df_model_next["Ladder_Position_Next"]

# Impute missing values
imputer = SimpleImputer(strategy="mean")
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)

# Train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_imputed, y_train)

# Prepare test data
test_df["Victorian"] = test_df["Victorian"].astype(int)
X_test = test_df[features].copy()
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# Predict
test_df["Ladder_Position_Next"] = model.predict(X_test_imputed)

print(test_df["Ladder_Position_Next"])

test_df[["ID", "Ladder_Position_Next"]].to_csv("submission.csv", index=False)


0       6.61
1       5.14
2       2.91
3       3.07
4       3.18
       ...  
469    13.41
470     7.50
471     8.56
472     9.36
473     8.68
Name: Ladder_Position_Next, Length: 474, dtype: float64
